# Examen bimestral

**Introducción a los Sistemas de Información**

Prof. Iván Carrera

Ingeniería en Ciencia de Datos e Inteligencia Artificial · Escuela Politécnica Nacional

**Calificación total:** 10 puntos.

## Caso de estudio · Papelería universitaria

Una papelería universitaria registra las compras de sus sucursales Central, Norte y Sur en un sistema de punto de venta. Cada compra genera un ticket que puede incluir varios productos. La persona encargada de la papelería necesita resumir estos registros para conocer qué se vendió, cómo se pagaron las compras y cómo variaron las ventas entre sucursales y meses.

Elabore los reportes solicitados a partir de los datos.

## Archivos de trabajo

| Archivo      | Contenido |
|--------------|---|
| `productos.csv` | Una fila por producto. Incluye su código, nombre, categoría y precio de lista. |
| `clientes.csv` | Una fila por cliente. Incluye su identificador, nombre, ciudad y tipo de cliente. |
| `ventas.csv`   | Una fila por producto dentro de un ticket. Incluye la fecha, sucursal, cliente, cantidad, precio unitario y método de pago. |

El campo `id_ticket` puede repetirse en varias filas de `ventas.csv`. Para relacionar los archivos, use `id_producto` en ventas y productos, y `ventas.id_cliente` con `clientes.id`. Un campo vacío en `metodo_pago` indica que ese dato quedó sin registrar.

## Preguntas

1. **Líneas, tickets y unidades.** ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

2. **Tickets por sucursal.** ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

3. **Métodos de pago.** ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

4. **Precios de lista por categoría.** ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

5. **Unidades por sucursal y categoría.** ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de `cantidad` en `ventas.csv`.

6. **Importe de ventas por sucursal.** ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como `cantidad × precio_unitario`.

7. **Tamaño de las compras.** ¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.

8. **Ventas por mes.** ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

9. **Producto más vendido por sucursal.** ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

10. **Compras por tipo de cliente.** Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

## Entrega y evaluación

Resuelva los reportes solicitados en un mismo Jupyter Notebook, identificados con el número de cada pregunta. Presente los reportes con encabezados claros y los importes en dólares con dos decimales. Use conteos, agrupaciones y cálculos descriptivos.

Entregue el link al Jupyter Notebook cargado en un repositorio Git en la Tarea del aula virtual.

**Cada pregunta vale 1 punto, para un total de 10 puntos.** En cada respuesta se evaluará que el código produzca el resultado solicitado a partir de los CSV y que se presenten todos los elementos pedidos en la pregunta.

Buena suerte.

In [ ]:
import pandas as pd

productos = pd.read_csv("/content/productos.csv")
clientes = pd.read_csv("/content/clientes.csv")
ventas = pd.read_csv("/content/ventas.csv")

**1. ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.**

In [13]:
import pandas as pd

pd.set_option("display.width", 120)
df = pd.read_csv("ventas.csv")

print(df.id_linea)


0       L00001
1       L00002
2       L00003
3       L00004
4       L00005
         ...  
2095    L02096
2096    L02097
2097    L02098
2098    L02099
2099    L02100
Name: id_linea, Length: 2100, dtype: str


Tiene 2100 líneas de venta.

In [15]:
print(df.id_ticket)

0       V0001
1       V0001
2       V0001
3       V0001
4       V0001
        ...  
2095    V0300
2096    V0300
2097    V0300
2098    V0300
2099    V0300
Name: id_ticket, Length: 2100, dtype: str


Hay 300 tickets distintos.

**2. ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?**

In [ ]:
print("\nTICKETS POR SUCURSAL")
tickets_sucursal = ventas.groupby("sucursal")["id_ticket"].nunique()
print(tickets_sucursal)
print("Sucursal con más tickets:", tickets_sucursal.idxmax())

**3. ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?**

In [27]:
print("\nMÉTODOS DE PAGO")
tickets_pago = ventas[["id_ticket", "metodo_pago"]].drop_duplicates()
print(tickets_pago["metodo_pago"].value_counts())


MÉTODOS DE PAGO


NameError: name 'ventas' is not defined

**4. ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.**

In [ ]:
print("\nPRECIOS DE LISTA POR CATEGORÍA")
precios_categoria = productos.groupby("categoria")["precio_lista"].agg(
    minimo="min",
    maximo="max",
    promedio="mean"
)
print(precios_categoria.round(2))

**5. ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de `cantidad` en `ventas.csv`.**

In [ ]:
print("\nUNIDADES POR SUCURSAL Y CATEGORÍA")
ventas_productos = ventas.merge(
    productos[["id_producto", "categoria"]],
    on="id_producto",
    how="left"
)

tabla = pd.pivot_table(
    ventas_productos,
    index="sucursal",
    columns="categoria",
    values="cantidad",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)

print(tabla)
total_ventas = ventas["cantidad"].sum()
total_tabla = tabla.loc["Total", "Total"]

print("\nCOMPROBACIÓN")
print("Total de unidades en ventas.csv:", total_ventas)
print("Total de unidades en tabla cruzada:", total_tabla)
print("¿Coinciden los totales?:", total_ventas == total_tabla)

**6. ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como `cantidad × precio_unitario`.**

In [26]:
ventas["importe_linea"] = ventas["cantidad"] * ventas["precio_unitario"]

print("IMPORTE DE VENTAS POR SUCURSAL")
importe_sucursal = ventas.groupby("sucursal")["importe_linea"].sum()
print(importe_sucursal)

NameError: name 'ventas' is not defined

**7. ¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.**

In [ ]:
print("\nTAMAÑO DE LAS COMPRAS")

unidades_ticket = ventas.groupby("id_ticket")["cantidad"].sum()

def clasificar_compra(unidades):
    if unidades <= 7:
        return "Hasta 7 unidades"
    elif unidades <= 14:
        return "De 8 a 14 unidades"
    else:
        return "15 o más unidades"

categorias_ticket = unidades_ticket.apply(clasificar_compra)

print(categorias_ticket.value_counts().reindex(
    ["Hasta 7 unidades", "De 8 a 14 unidades", "15 o más unidades"],
    fill_value=0
))


**8. ¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.**

In [ ]:
print("\nVENTAS POR MES")

ventas["mes"] = ventas["fecha"].dt.to_period("M")

ventas_mes = ventas.groupby("mes").agg(
    tickets=("id_ticket", "nunique"),
    importe_total=("importe_linea", "sum")
)

print(ventas_mes)


mes_mayor = ventas_mes["importe_total"].idxmax()

print("\nMES CON MAYOR IMPORTE")
print("Mes:", mes_mayor)
print("Importe:", ventas_mes.loc[mes_mayor, "importe_total"])